# Residuals, uncertainty and identifiability

**Level 2 · Prerequisites: Inverse problems: evidence and non-uniqueness; reading time: 25 minutes.**

## Objectives

Separate signed residuals, normalized error, uncertainty assumptions and model non-uniqueness. The committed product result contains train/holdout metrics and multiple accepted mathematical-depth alternatives; inspect those observations without reconstructing them.

## Setup

Use the repository's reviewed Python 3.12 notebook environment. This notebook installs nothing. The artifact is a synthetic, product-generated prism case. Error values are stored summaries; no observation arrays are fabricated in this lesson.

The following cell reads the saved train/holdout metrics and non-uniqueness ledger.

In [ ]:
import json
from pathlib import Path

# Read computed product metrics and alternative models; do not synthesize observations.
root = Path.cwd().parents[2]
case = root / 'data/derived/m01-scientific-course/prism-case-0'
result = json.loads((case / 'result.json').read_text(encoding='utf-8'))
for split in ('train', 'holdout'):
    print(split, result['evaluation'][split])
print('uncertainty contract:', result['uncertainty'])
print('training-only alternatives:')
for model in result['nonuniqueness']['training_only_alternatives']:
    print({k: model[k] for k in ('depth_m', 'damping', 'training_rmse_mgal', 'status')})

## Data

The artifact is a synthetic, product-generated prism case. Error values are stored summaries; no observation arrays are fabricated in this lesson.

**Files:** `data/derived/m01-scientific-course/prism-case-0/result.json`, `receipt.json`.

## Method

For residual r=d−F(m), RMSE has data units while normalized RMSE divides by the declared noise scale. A normalized score near one is not sufficient if observations are unsupported or correlated. The reported uncertainty is conditional linear propagation with independent-station diagonal weights; it excludes geometry, parameter selection, bias and geological uncertainty. The forward-model and parameter-estimation context is described by SimPEG ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); the numerical metrics here are from the product artifact.

For residual r=d−F(m), diagonal Gaussian errors give φ_d=½∑ᵢ(rᵢ/σᵢ)². The standardized residual rᵢ/σᵢ is dimensionless; raw RMSE retains the data unit. If errors are correlated, use rᵀC_d⁻¹r rather than a diagonal sum. The product case explicitly assumes independent station errors and excludes geometry error, selection uncertainty, bias and geological uncertainty, so its propagated σ is conditional rather than a posterior geological interval. This distinction is central to inverse parameter estimation [Cockett et al. (2015)](https://doi.org/10.1016/j.cageo.2015.09.015).

Spatial holdout matters because nearby potential-field observations share smooth responses. Randomly withholding isolated points can leave near-duplicates in training and exaggerate transfer skill. This stored case has 149 training rows, 43 outer-holdout rows and four masked rows; only 30 holdout predictions are covered. Its normalized RMSE is about 0.735 on train and 1.289 on covered holdout, while 13 holdout rows lack support. Alternative mathematical depths of 160, 360 and 700 m have different damping and fit. A “passed” candidate means a bounded numerical gate passed; depth is a basis parameter, not a physical source depth.

## Results

The stored holdout normalized RMSE exceeds train RMSE, and 13 holdout rows are unsupported. Several training-only depth alternatives are marked passed with different damping and fit. This is evidence of conditional ambiguity, not permission to select a preferred geology. The output must be interpreted alongside its data support and uncertainty exclusions.

These panels compare the product’s saved blocked-CV candidates against outer evaluation. Failed candidates have no score and are omitted rather than assigned an invented penalty. The plot is a diagnostic of the authored synthetic case.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Plot actual persisted cross-validation candidates and train/holdout metrics without regenerating the fit.
p = Path.cwd().parents[2] / 'data/derived/m01-scientific-course/prism-case-0/result.json'
a = json.loads(p.read_text(encoding='utf-8'))
fig, (ax,bx) = plt.subplots(1,2,figsize=(11,4.8),constrained_layout=True)
for damp in sorted({c['damping'] for c in a['selection']['candidates']}):
 cs=[c for c in a['selection']['candidates'] if c['damping']==damp and c.get('cv_normalized_rmse') is not None]
 ax.plot([c['depth_m'] for c in cs],[c['cv_normalized_rmse'] for c in cs],marker='o',label=f'damping={damp:g}')
ax.set(xlabel='Harmonic basis depth [m]',ylabel='Blocked-CV normalized RMSE [1]',title='Stored candidate scores'); ax.legend()
labels=['Train','Covered holdout']; vals=[a['evaluation'][k]['normalized_rmse'] for k in ('train','holdout')]
bx.bar(labels,vals,color=['#4878a8','#d65f5f']); bx.set(ylabel='Normalized RMSE [1]',title='Outer evaluation')
for i,k in enumerate(('train','holdout')):
 m=a['evaluation'][k]; bx.text(i,vals[i],f"n={m['covered_count']} covered; {m['unsupported_count']} unsupported",ha='center',va='bottom',fontsize=8)
print('weighted design condition:',a['condition']['weighted_design_condition'])
print('uncertainty excludes:',a['uncertainty']['excludes'])
plt.show()


## Summary

A small residual is a statement about predictions under one experiment and weighting scheme. It is not proof of a unique model or calibrated geological uncertainty.


## References

- https://doi.org/10.1016/j.cageo.2015.09.015